# 🍽️ Personal Chef Agent — Live Demo

An AI-powered personal chef built with **LangChain + LangGraph + Tavily + Groq**.

Give it your leftover ingredients — it searches the web for real recipes and holds a full multi-turn conversation, remembering everything you said.

---

**Tech stack:**
- `create_agent()` — LangGraph agent abstraction  
- `@tool` + Tavily — real-time web search  
- `InMemorySaver` — short-term conversational memory via checkpointing  
- `groq:openai/gpt-oss-120b` — fast, free-tier inference on Groq

**Course:** Introduction to LangChain — Python (LangChain Academy, Module 1)  
**Author:** Muhammad Ali Rehman

---
## 1. Setup

In [1]:
from dotenv import load_dotenv

load_dotenv()  # loads GROQ_API_KEY, TAVILY_API_KEY from .env

True

---
## 2. Define the web search tool

The `@tool` decorator turns any Python function into something the agent can call autonomously.
- The **function name** becomes the tool name the model sees.
- The **docstring** becomes the description — this is what tells the model *when* to use this tool.

In [2]:
from langchain.tools import tool
from typing import Dict, Any
from tavily import TavilyClient

tavily_client = TavilyClient()

@tool
def web_search(query: str) -> Dict[str, Any]:
    """Search the web for up-to-date recipes and cooking information."""
    return tavily_client.search(query)

---
## 3. System prompt

The system prompt sets the agent's **persona, role, and rules of engagement** before any user message arrives.

In [3]:
system_prompt = """
You are a personal chef assistant. The user will tell you what leftover
ingredients they have at home.

Your job:
1. Use the web_search tool to find real recipes that can be made with
   those ingredients.
2. Return 2–3 clear recipe suggestions, each with a short ingredient list
   and quick cooking steps.
3. If the user asks for more detail on any recipe, provide full
   step-by-step instructions.
4. Answer any follow-up cooking questions using the conversation history.

Keep your responses practical, friendly, and concise.
"""

---
## 4. Build the agent

Three things make this agent powerful:
- **`tools=[web_search]`** — gives it the ability to act beyond its training knowledge
- **`system_prompt`** — shapes its persona and behaviour
- **`checkpointer=InMemorySaver()`** — persists the conversation state so it remembers previous messages

In [4]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model="groq:openai/gpt-oss-120b",
    tools=[web_search],
    system_prompt=system_prompt,
    checkpointer=InMemorySaver(),
)

---
## 5. Turn 1 — Ask for recipe ideas

The `thread_id` ties all invocations to the same conversation.
The same `config` is passed on every turn — that's how the agent remembers.

In [5]:
from langchain.messages import HumanMessage

# Thread ID groups all messages into one conversation
config = {"configurable": {"thread_id": "chef-demo-1"}}

response = agent.invoke(
    {"messages": [HumanMessage(content="I have some leftover chicken and rice. What can I make?")]},
    config,
)

print(response["messages"][-1].content)

Here are a few tasty ways to turn your leftover chicken + rice into a brand‑new meal. I've pulled the most‑liked, quick‑to‑make recipes that use only pantry staples (plus a few optional veggies or sauces you probably already have). Click the links for the full recipe pages, and I've also included a short "at‑a‑glance" ingredient list and the basic steps for each one.

---

## 1️⃣ Chicken Fried Rice (30‑minute skillet dinner)

**Why it works:**  
- Uses cold, day‑old rice (the grains stay separate).  
- You can toss in whatever frozen veggies you have.  
- One‑pan, so cleanup is minimal.

**Core ingredients** (you'll need about 2 cups cooked rice and 1½ – 2 cups diced cooked chicken):

| Ingredient | Amount |
|------------|--------|
| Olive oil (or any neutral oil) | 1 tsp |
| Sesame oil (optional, for flavor) | 1 tsp |
| Frozen peas + carrots mix | 1 cup |
| Green onions, sliced | 4 (divide) |
| Garlic, minced | 2 cloves |
| Eggs, lightly whisked | 2 |
| Cold cooked rice | 3 cups |
| C

---
## 6. Under the hood — the ReAct message flow

Let's inspect every message the agent exchanged to see the full ReAct loop:

```
HumanMessage       → user's question
AIMessage          → agent decides to call web_search (tool_calls field, empty content)
ToolMessage        → Tavily returns live search results
AIMessage          → agent synthesises the final answer
```

In [6]:
for i, msg in enumerate(response["messages"], 1):
    msg_type = type(msg).__name__
    if hasattr(msg, "tool_calls") and msg.tool_calls:
        print(f"Message {i} | {msg_type}  (tool call — content is empty, action is in tool_calls)")
        print(f"  → tool_calls: {msg.tool_calls}")
    elif msg_type == "ToolMessage":
        preview = str(msg.content)[:120] + "..."
        print(f"Message {i} | {msg_type}  (Tavily search result returned to agent)")
        print(f"  → '{preview}'")
    else:
        preview = str(msg.content)[:80] + ("..." if len(str(msg.content)) > 80 else "")
        print(f"Message {i} | {msg_type}")
        print(f"  → '{preview}'")
    print()

Message 1 | HumanMessage
  → 'I have some leftover chicken and rice. What can I make?'

Message 2 | AIMessage  (tool call — content is empty, action is in tool_calls)
  → tool_calls: [{'name': 'web_search', 'args': {'query': 'leftover chicken and rice recipes'}, 'type': 'tool_call'}]

Message 3 | ToolMessage  (Tavily search result returned to agent)
  → '{"query": "leftover chicken and rice recipes", "results": [{"url": "https://..."...}]}'

Message 4 | AIMessage  (final synthesised answer)
  → 'Here are a few tasty ways to turn your leftover chicken + rice...'


---
## 7. Turn 2 — Follow-up (memory test)

The agent uses the **same `config`** (same `thread_id`), so the `InMemorySaver` checkpointer replays all previous messages.
No context is re-sent by us — the agent retrieves it automatically.

In [7]:
# Same config = same thread = agent remembers the first message
response2 = agent.invoke(
    {"messages": [HumanMessage(content="Give me the full recipe for the first one")]},
    config,  # same thread_id — agent has full memory of turn 1
)

print(response2["messages"][-1].content)

Here are the full step-by-step instructions for **Chicken Fried Rice**:

### 🍳 Chicken Fried Rice — Full Instructions

**Prep time:** 5 min | **Cook time:** 10 min | **Total:** 15 min | **Serves:** 4

**Ingredients:**
- 1 tsp olive oil
- 1 tsp sesame oil
- 1 cup frozen peas and carrots mix
- 4 green onions, sliced (divided)
- 2 cloves garlic, minced
- 2 eggs, lightly whisked
- 3 cups cold cooked white rice (day-old works best)
- 1½ cups diced cooked chicken
- 3 tbsp soy sauce (or tamari for gluten-free)
- 1 tbsp hoisin or oyster sauce (optional)
- Salt & pepper to taste
- Sriracha or chili oil (optional, for heat)

**Instructions:**

1. **Heat the pan** — Add olive oil and sesame oil to a large skillet or wok over medium-high heat.
2. **Veggies** — Add frozen peas-carrots mix and half the green onions. Stir-fry 2–3 min until veggies are thawed and slightly browned.
3. **Aromatics** — Add garlic and stir for 30 seconds until fragrant.
4. **Eggs** — Push everything to one side. Pour whis

---
## 8. Turn 3 — Memory verification

Let's confirm the agent truly remembers the full conversation by asking something that requires context from both previous turns.

In [8]:
response3 = agent.invoke(
    {"messages": [HumanMessage(content="What ingredients did I say I had, and what recipes did you suggest?")]},
    config,
)

print(response3["messages"][-1].content)

You gave me **leftover chicken and rice**. Based on that, I suggested three recipes:

1. **Chicken Fried Rice** (the one we just went through in detail)
2. **One-Pot Chicken & Rice Casserole** — creamy baked dish with cream of chicken soup and cheese
3. **Simple One-Pot Chicken & Rice** — stovetop, Asian-inspired with soy sauce and ginger

Would you like the full recipe for option 2 or 3 as well?


---
## 9. How it all works — Architecture

```
User Input  (ingredients or question)
      │
      ▼
 create_agent()
 ├── model:       groq:openai/gpt-oss-120b
 ├── tools:       [web_search]  ← Tavily real-time search
 ├── system_prompt: "You are a personal chef..."
 └── checkpointer: InMemorySaver()  ← persists state per thread_id
      │
      ▼
 ReAct Loop:
   [Reason]  Agent decides: call web_search(query)
      │
      ▼
   [Act]     web_search() hits Tavily API → live results returned
      │
      ▼
   [Observe]  ToolMessage carries results back to agent
      │
      ▼
   [Respond] AIMessage with final synthesised answer
      │
      ▼
 InMemorySaver snapshots the full state
 → Next turn picks up from the same thread_id
```

---

## 10. Key concepts demonstrated

| Concept | How it appears in this project |
|---------|--------------------------------|
| `create_agent()` | Core agent builder — wraps model, tools, prompt, memory |
| `@tool` decorator | Turns `web_search()` into an agent-callable action |
| ReAct loop | Agent reasons → calls `web_search` → observes result → answers |
| Tavily Search | Provides real, up-to-date recipe results the model doesn't know |
| `InMemorySaver` | Checkpointer that saves state snapshots after every turn |
| Thread ID | Groups all turns into one persistent conversation |
| System prompt | Sets the chef persona and defines behavioural rules |
| `HumanMessage` | Typed message wrapper for user input |
| `response['messages'][-1]` | Access the final AI response from the state |
